# Data Cleaning Workflow

This notebook starts the reproducible data-cleaning workflow for the selected targets and the predictor datasets listed in the literature CSV. Reusable download logic lives in `2_data/scripts/raw_data_download.py`; this notebook imports the script, runs the workflow, and displays the resulting manifest.

## Optional Colab Setup

This cell is safe for local use. It only performs setup when the notebook runs inside Google Colab.

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Colab detected. Mount your repository and set ROOT_DIR in the next cell if needed.")
else:
    print("Local environment detected; skipping Colab setup.")

Local environment detected; skipping Colab setup.


## Repository Setup

In [2]:
from pathlib import Path
import sys


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "0_organization").exists() and (path / "2_data").exists():
            return path
    raise RuntimeError("Could not find repository root. Run this notebook from inside the repository.")


ROOT_DIR = find_repo_root(Path.cwd())
SCRIPT_DIR = ROOT_DIR / "2_data" / "scripts"

if str(SCRIPT_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPT_DIR))

ROOT_DIR

PosixPath('/Users/marcjin/Desktop/Studium/Master/26SS/env-innovation-prediction')

## Literature CSV-Based Raw Download Plan

The predictor download plan uses `1_literature_review/Managerial AI- literature review - List 1.csv` as the primary source. The manifest is generated by `run_raw_download`; do not edit it by hand. If a future literature source lacks a machine-readable endpoint, extend `2_data/scripts/raw_data_download.py` so the missing source remains reproducible.

In [3]:
import pandas as pd
from data_common import RAW_PREDICTORS_V1_DIR
from raw_data_download import (
    LITERATURE_CSV_PATH,
    build_literature_predictor_download_plan,
    build_raw_download_plan,
    run_raw_download,
)

START_YEAR = 1990
END_YEAR = 2024
RAW_OUTPUT_DIR = RAW_PREDICTORS_V1_DIR

literature_plan = pd.DataFrame(
    build_literature_predictor_download_plan(LITERATURE_CSV_PATH, START_YEAR, END_YEAR)
)
download_plan = pd.DataFrame(build_raw_download_plan(START_YEAR, END_YEAR, LITERATURE_CSV_PATH))

literature_plan[[
    "dataset_id",
    "variable",
    "source_variable",
    "status",
    "download_method",
    "literature_rows",
    "literature_predictors",
]]

,dataset_id,variable,source_variable,status,download_method,literature_rows,literature_predictors
0,world_bank_wdi,rd_expenditure_gdp,GB.XPD.RSDV.GD.ZS,planned,tabular,3,R&D expenditure / GDP
1,world_bank_wdi,researchers_per_million,SP.POP.SCIE.RD.P6,planned,tabular,4,Researchers / human capital
2,world_bank_wdi,tertiary_enrollment,SE.TER.ENRR,planned,tabular,5,Tertiary enrollment
3,world_bank_wdi,scientific_journal_articles,IP.JRN.ARTC.SC,planned,tabular,6,Scientific journal articles
4,world_bank_wdi,high_tech_exports,TX.VAL.TECH.MF.ZS,planned,tabular,7,High-tech exports
5,world_bank_wdi,resident_patent_applications,IP.PAT.RESD,planned,tabular,8,Patent applications (total) - output
6,oecd_patents_environment,env_technology_share_for_rta,PT_TECH.DEV.ENV_PAT._Z,planned,tabular,9,Lagged Env Tech RTA
7,oecd_patents_environment,env_co_invention_share,PT_TECH_COL.COL.ENV_PAT._Z,planned,tabular,10,Co-invention Rate -> International Collaboration
8,world_bank_wdi,renewable_energy_share,EG.FEC.RNEW.ZS,planned,tabular,12,Renewable energy share
9,world_bank_wdi,co2_per_capita_ar5,EN.GHG.CO2.PC.CE.AR5,planned,tabular,13,CO2 emission per capital


## Download Raw Files

Running this cell writes source files to `2_data/raw/predictorsv1/` and writes `raw_download_manifest.csv` in the same directory with source URLs, source status, download dates, file paths, and source notes. Both the raw files and manifest are code-generated artifacts.

In [4]:
manifest = run_raw_download(start_year=START_YEAR, end_year=END_YEAR, raw_dir=RAW_OUTPUT_DIR)
manifest[[
    "dataset_id",
    "variable",
    "role",
    "status",
    "rows",
    "columns",
    "file_format",
    "file_path",
    "notes",
]]

,dataset_id,variable,role,status,rows,columns,file_format,file_path,notes
0,oecd_patents_environment,env_patent_share_inventions,main_target,downloaded,3804,26,csv,/Users/marcjin/Desktop/Studium/Master/26SS/env...,
1,oecd_patents_environment,env_patents_per_million,robustness_target,downloaded,3727,26,csv,/Users/marcjin/Desktop/Studium/Master/26SS/env...,
2,world_bank_wdi,rd_expenditure_gdp,literature_predictor,downloaded,9310,7,csv,/Users/marcjin/Desktop/Studium/Master/26SS/env...,related with Environmental policy stringency (...
3,world_bank_wdi,researchers_per_million,literature_predictor,downloaded,9310,7,csv,/Users/marcjin/Desktop/Studium/Master/26SS/env...,
4,world_bank_wdi,tertiary_enrollment,literature_predictor,downloaded,9310,7,csv,/Users/marcjin/Desktop/Studium/Master/26SS/env...,"WGI - Source, recovery?"
5,world_bank_wdi,scientific_journal_articles,literature_predictor,downloaded,9310,7,csv,/Users/marcjin/Desktop/Studium/Master/26SS/env...,
6,world_bank_wdi,high_tech_exports,literature_predictor,downloaded,9310,7,csv,/Users/marcjin/Desktop/Studium/Master/26SS/env...,
7,world_bank_wdi,resident_patent_applications,literature_predictor,downloaded,9310,7,csv,/Users/marcjin/Desktop/Studium/Master/26SS/env...,
8,oecd_patents_environment,env_technology_share_for_rta,literature_predictor_derived_source,downloaded,3804,26,csv,/Users/marcjin/Desktop/Studium/Master/26SS/env...,past specialization predicts future innovation...
9,oecd_patents_environment,env_co_invention_share,literature_predictor_derived_source,downloaded,2091,26,csv,/Users/marcjin/Desktop/Studium/Master/26SS/env...,Because domestic innovation capacity alone mis...


## Download Status Checks

The status table is a reproducibility check. A successful raw download run should have only `downloaded` rows; any future non-downloaded row means the downloader should be extended before downstream cleaning proceeds.

In [5]:
status_counts = (
    manifest.groupby(["status", "dataset_id"], dropna=False)
    .size()
    .reset_index(name="entries")
    .sort_values(["status", "dataset_id"])
)

unresolved_sources = manifest[manifest["status"].ne("downloaded")][[
    "variable",
    "source",
    "source_url",
    "status",
    "notes",
]]

display(status_counts)
display(unresolved_sources)

,status,dataset_id,entries
0,downloaded,oecd_carbon_pricing,2
1,downloaded,oecd_environment_tax,1
2,downloaded,oecd_eps,1
3,downloaded,oecd_patents_environment,4
4,downloaded,policy_uncertainty,1
5,downloaded,world_bank_data360,1
6,downloaded,world_bank_wdi,16
7,downloaded,world_bank_wgi,1


,variable,source,source_url,status,notes


## Next Step

The next cleaning step should read the downloaded raw files, standardize country-year columns, construct three-year lagged moving averages for predictors, and write a processed modeling panel to `2_data/processed/`. Future source gaps should be resolved in the downloader rather than by manually editing raw data or the manifest.